# R7.3 — 2023 年 ADT 不完整流轴六断面独立物理 QC（纯 Python）

**目的**：对 R7.2 已经保存的 274 条失败前部分流轴进行**卫星 ADT 本身**的物理/几何质量审查，确定 A、B、PN、C、D、E 哪些交点可进入**标注为敏感性分析**的附加时间序列。

**科学合同**：不改变原追踪器；不读取 GLORYS 来调整筛选标准或交点位置；阈值仅从 2018–2022 年已经接受的 ADT 流轴上标定。未通过 QC 的位置、终止过近、断面多交点均拒绝；保留全部拒绝记录。部分流轴不视为完整参考流轴，不改变 R7.0 完整追踪成功率或 R7.2 10,279 天整体 RMSE。预测 vs ADT 仍须独立的真实 checkpoint 预测路径。

> 在已有 R7.0 + R7.2 结果和 1993–2023 NetCDF 均存在的情况下，执行 Run All 即可。优先仅扫描 2018–2022 参考样本和 2023 年 274 天，不重跑 31 年。

In [ ]:
# Cell 0 — 路径、依赖与实验开关
from pathlib import Path
import os, io, re, json, zipfile, hashlib, traceback, time
import numpy as np, pandas as pd
from scipy.ndimage import gaussian_filter, map_coordinates
from IPython.display import display, FileLink
import matplotlib.pyplot as plt
import xarray as xr

ROOT=Path(os.environ.get('KUROSHIO_ROOT','/mnt/d/KuroshioDaily1993_2025'))
R70=ROOT/'R70_ADT_Independent_Python'
ADT_FULL=R70/'adt_axes_npz'
GLORYS=ROOT/'final/tracks_final' # 仅用在最终QC冻结之后的附加误差统计
PART=ROOT/'R72_ADT_CompleteDomain_2023PartialAudit'/'adt_partial_2023'
R72_ZIP=ROOT/'R72_上传给ChatGPT_全量共域与2023断面审计.zip'
R70_ZIP=ROOT/'R70_PY_上传给ChatGPT_ADT独立参考验证完整结果.zip'
ADT_ROOT=Path('/mnt/f/01-CMEMS/SEALEVEL_GLO_PHY_L4_MY_008_047/cmems_obs-sl_glo_phy-ssh_my_allsat-l4-duacs-0.125deg_P1D')
OUT=ROOT/'R73_ADT_2023_PartialPhysicalQC'
TBL=OUT/'tables'; REP=OUT/'reports'; FIG=OUT/'figures_en';CACHE=OUT/'adt_qc_cache'
for p in (OUT,TBL,REP,FIG,CACHE):p.mkdir(parents=True,exist_ok=True)
CALIBRATION_YEARS=range(2018,2023)
CALIBRATION_DAYS_PER_YEAR=15   # 75日期、至多450断面样本；仅ADT，全量主结果不受影响
FORCE_QC_REPROCESS=False
RIDGE_HALF_WIDTH_KM=25.
RIDGE_SAMPLES=21
THRESHOLD_QUANTILE=0.05       # 2018-22完整ADT参考分布的下5%分位；非依据2023 GLORYS误差调整
MIN_CALIBRATION_PER_SECTION=30
USE_GLORYS_AFTER_FROZEN_QC=True
REQUIRE_UNIQUE_INTERSECTION=True
MIN_DOWNSTREAM_POINTS=3
SECTIONS={
 'A':((121.50,23.50),(125.00,22.75)),
 'B':((121.50,26.25),(124.25,24.25)),
 'PN':((126.00,29.00),(128.15,27.30)),
 'C':((130.25,31.25),(132.50,27.00)),
 'D':((137.00,34.75),(138.00,27.75)),
 'E':((141.75,40.00),(145.00,31.50)),
}
for label,p in [('R72报告',R72_ZIP),('R70报告',R70_ZIP),('ADT原始NetCDF',ADT_ROOT),('2023部分轴',PART),('2018–2022完整ADT轴',ADT_FULL)]:
 print(label,p.exists(),p)
 if not p.exists():raise FileNotFoundError(f'{label}缺失：{p}')
print('2023部分NPZ',len(list(PART.glob('2023*.npz'))))


In [ ]:
# Cell 1 — 加载R7.2候选与R7.0完整ADT断面，严格复核源数据版本

def read_zip_csv(path,inner):
 with zipfile.ZipFile(path) as z:
  assert z.testzip() is None
  return pd.read_csv(io.BytesIO(z.read(inner)))
part_status=read_zip_csv(R72_ZIP,'tables/06_2023部分ADT轨迹状态.csv')
part_sec=read_zip_csv(R72_ZIP,'tables/07_2023部分ADT轨迹六断面候选.csv')
full_sections=read_zip_csv(R70_ZIP,'daily/每日六断面交点和多交点信息.csv')
full_sections=full_sections[full_sections.source=='ADT'].copy()
assert set(part_sec.section)==set(SECTIONS), 'R72断面表与六断面定义不匹配'
assert len(part_status)==274 and part_sec.date.nunique()==274, '结果并非R7.2完整274日版本'
assert len(part_sec)==274*6, '缺少部分轨迹的断面候选行'
assert part_sec.duplicated(['date','section']).sum()==0
assert len(set(full_sections.date)&set(part_status.date))==0, '原完整参考与失败日重叠，请核查'
print('R7.2:',len(part_status),'不完整日期;',len(part_sec),'断面候选; R7.0完整ADT交点:',len(full_sections))


In [ ]:
# Cell 2 — ADT-only物理场：与MATLAB V6一致的细/粗尺度NaN感知Gaussian和水平梯度 (m/km)
# NetCDF仅加载[120,150]E×[20,40]N区域，验证时间戳并防止自动选错日期。

def nc_for_date(day):
 day=pd.Timestamp(day);year=str(day.year);ymd=day.strftime('%Y%m%d')
 fs=sorted((ADT_ROOT/year).glob(f'dt_global_allsat_phy_l4_{ymd}_*.nc'))
 if not fs:raise FileNotFoundError(f'找不到 {day.date()} 对应ADT NC')
 return fs[-1] # 同日多修订版必须记录来源；本次不依据GLORYS选择版本

def read_adt_field(path,day):
 with xr.open_dataset(path,decode_times=True,mask_and_scale=True) as ds:
  names=list(ds.variables)
  lonkey=next((k for k in ('longitude','lon') if k in names),None)
  latkey=next((k for k in ('latitude','lat') if k in names),None)
  key=next((k for k in ('adt','ADT') if k in names),None)
  if not all((lonkey,latkey,key)):raise KeyError(f'ADT坐标或变量缺失 {names}')
  z=ds[key]
  for coord,low,high in ((lonkey,120.,150.),(latkey,20.,40.)):
   a=np.asarray(ds[coord].values,float)
   if a.ndim!=1:raise ValueError('本Notebook仅处理规则经纬网格')
   z=z.sel({coord:slice(low,high) if a[0]<a[-1] else slice(high,low)})
  for dim in tuple(z.dims):
   if dim in (lonkey,latkey):continue
   if dim in ds.coords:
    vals=pd.to_datetime(ds[dim].values)
    i=int(np.argmin(np.abs(vals-pd.Timestamp(day))))
    if abs(pd.Timestamp(vals[i])-pd.Timestamp(day))>pd.Timedelta(hours=18):raise ValueError(f'{day}: NC时间不匹配 {vals[i]}')
    z=z.isel({dim:i})
   elif z.sizes[dim]==1:z=z.isel({dim:0})
   else:raise ValueError(f'未知非单一维度 {dim}')
  z=z.transpose(latkey,lonkey)
  lo=np.asarray(z[lonkey].values,float);la=np.asarray(z[latkey].values,float);a=np.asarray(z.values,float)
  if lo[0]>lo[-1]:lo=lo[::-1];a=a[:,::-1]
  if la[0]>la[-1]:la=la[::-1];a=a[::-1]
  a[np.abs(a)>100]=np.nan
  if len(lo)<10 or len(la)<10:raise ValueError('ROI格点数不足')
  return lo,la,a

def nan_gauss(a,sigma):
 v=np.isfinite(a).astype(float);v0=np.where(np.isfinite(a),a,0.)
 den=gaussian_filter(v,sigma,mode='nearest')
 o=gaussian_filter(v0,sigma,mode='nearest')/np.maximum(den,1e-9)
 o[den<.60]=np.nan
 return o

def fields(lo,la,adt):
 ff=nan_gauss(adt,1.);cc=nan_gauss(adt,3.)
 la2d=np.broadcast_to(la[:,None],ff.shape)
 # np.gradient接受真实坐标数组，保证0.125°像元与经纬长度转换正确
 fy,fx=np.gradient(ff,la,lo);cy,cx=np.gradient(cc,la,lo)
 kmx=111.32*np.cos(np.deg2rad(la2d))
 return {'lon':lo,'lat':la,'fine':np.hypot(fx/kmx,fy/111.32),
         'coarse':np.hypot(cx/kmx,cy/111.32),'gx':fx/kmx,'gy':fy/111.32}

def interp2(F,name,lon,lat):
 lo=F['lon'];la=F['lat'];x=np.asarray(lon,float);y=np.asarray(lat,float)
 # regular 0.125° coordinates -> use map_coordinates for vectorized bilinear interpolation
 xi=np.interp(x,lo,np.arange(len(lo)),left=np.nan,right=np.nan)
 yi=np.interp(y,la,np.arange(len(la)),left=np.nan,right=np.nan)
 coords=np.array([np.nan_to_num(yi,nan=-1e6),np.nan_to_num(xi,nan=-1e6)])
 return map_coordinates(F[name],coords,order=1,mode='constant',cval=np.nan)

def load_axis(p):
 with np.load(p,allow_pickle=False) as z:
  if 'axis_lon' in z.files: return np.column_stack([z['axis_lon'].reshape(-1),z['axis_lat'].reshape(-1)]).astype(float)
  if 'lon' in z.files:return np.column_stack([z['lon'].reshape(-1),z['lat'].reshape(-1)]).astype(float)
  raise KeyError(f'未知NPZ流轴keys: {z.files}')

# 近似km局地几何只用于寻找切线和局地剖面，不替代最终测地学误差

def at_km(x,ref_lat):
 a=np.asarray(x,float)
 return np.stack([a[...,0]*111.32*np.cos(np.deg2rad(ref_lat)),a[...,1]*111.32],axis=-1)

def section_qc_physics(F,axis,lon,lat):
 """候选点处ADT梯度值与±25km法向剖面之最大梯度比，以及梯度对应的地转流向和轨迹切向一致性。"""
 a=np.asarray(axis,float);point=np.array([float(lon),float(lat)])
 xy=at_km(a,lat);p=at_km(point,lat)
 idx=int(np.argmin(np.linalg.norm(xy-p,axis=1)))
 i0=max(idx-1,0);i1=min(idx+1,len(a)-1)
 tang=xy[i1]-xy[i0];norm=np.linalg.norm(tang)
 if norm<1e-6:raise ValueError('断面交点附近切线无效')
 tang/=norm;normal=np.array([-tang[1],tang[0]])
 xx=np.linspace(-RIDGE_HALF_WIDTH_KM,RIDGE_HALF_WIDTH_KM,RIDGE_SAMPLES)
 xlon=lon+xx*normal[0]/(111.32*np.cos(np.deg2rad(lat)))
 ylat=lat+xx*normal[1]/111.32
 f=interp2(F,'fine',xlon,ylat);c=interp2(F,'coarse',xlon,ylat)
 f0=float(interp2(F,'fine',np.array([lon]),np.array([lat]))[0]);c0=float(interp2(F,'coarse',np.array([lon]),np.array([lat]))[0])
 gx=float(interp2(F,'gx',np.array([lon]),np.array([lat]))[0]);gy=float(interp2(F,'gy',np.array([lon]),np.array([lat]))[0])
 tg=np.array([-gy,gx]);tg_norm=np.linalg.norm(tg)
 if tg_norm<1e-9:raise ValueError('ADT梯度为0或不可用')
 fr=f0/max(float(np.nanmax(f)),1e-10) if np.isfinite(f).any() else np.nan
 cr=c0/max(float(np.nanmax(c)),1e-10) if np.isfinite(c).any() else np.nan
 return {'fine_ridge_ratio':float(fr),'coarse_ridge_ratio':float(cr),'geostrophic_dir_cos':float(np.dot(tang,tg/tg_norm)),
         'fine_grad_mkm':f0,'coarse_grad_mkm':c0,'nearest_axis_index':idx}


In [ ]:
# Cell 3 — 用2018–2022已完整接受的ADT流轴建立独立QC参照阈值（不使用GLORYS）
CAL_FILE=TBL/'01_2018_2022_ADT基准抽样物理支持.csv'
if CAL_FILE.exists() and not FORCE_QC_REPROCESS:
 baseline=pd.read_csv(CAL_FILE)
else:
 records=[]
 for year in CALIBRATION_YEARS:
  days=sorted(full_sections.loc[(full_sections.date.astype(str).str[:4]==str(year)) & (full_sections.section=='D'),'date'].astype(str).unique())
  if len(days)<CALIBRATION_DAYS_PER_YEAR:raise ValueError(f'基准年 {year} 不足{CALIBRATION_DAYS_PER_YEAR}天')
  chosen=[days[int(i)] for i in np.unique(np.linspace(0,len(days)-1,CALIBRATION_DAYS_PER_YEAR,dtype=int))]
  for day in chosen:
   rows=full_sections[(full_sections.date==day)&(full_sections.n_intersections==1)]
   if not len(rows):continue
   try:
    path=nc_for_date(day);lo,la,adt=read_adt_field(path,day);F=fields(lo,la,adt)
    axis=load_axis(ADT_FULL/(day.replace('-','')+'.npz'))
    for r in rows.itertuples(index=False):
     if not np.isfinite(r.lon) or not np.isfinite(r.lat):continue
     try:
      obs=section_qc_physics(F,axis,r.lon,r.lat)
      records.append({'date':day,'section':r.section,'source_nc':str(path),'error':'',**obs})
     except Exception as e:records.append({'date':day,'section':r.section,'error':repr(e)})
   except Exception as e:
    for s in SECTIONS:records.append({'date':day,'section':s,'error':repr(e)})
  print('reference',year,'done')
 baseline=pd.DataFrame(records)
 baseline.to_csv(CAL_FILE,index=False,encoding='utf-8-sig')
THRESHOLDS={}
for sec in SECTIONS:
 q=baseline[baseline.section==sec].dropna(subset=['fine_ridge_ratio','coarse_ridge_ratio','geostrophic_dir_cos'])
 if len(q)<MIN_CALIBRATION_PER_SECTION:raise RuntimeError(f'ADT校准断面{sec}有效样本仅 {len(q)}，拒绝猜阈值；检查F盘时间/NetCDF')
 THRESHOLDS[sec]={k:float(np.quantile(q[k],THRESHOLD_QUANTILE)) for k in ['fine_ridge_ratio','coarse_ridge_ratio','geostrophic_dir_cos']}
 THRESHOLDS[sec]['calibration_n']=int(len(q))
print('Frozen thresholds (ADT-only):')
display(pd.DataFrame(THRESHOLDS).T.round(3))
(REP/'R73_预先冻结的ADT物理QC阈值.json').write_text(json.dumps({'years':list(CALIBRATION_YEARS),'days_per_year':CALIBRATION_DAYS_PER_YEAR,'quantile':THRESHOLD_QUANTILE,'thresholds':THRESHOLDS},ensure_ascii=False,indent=2),encoding='utf8')


In [ ]:
# Cell 4 — 对274天全部不完整路径做ADT-only QC：记录拒绝原因，断面多交点默认拒绝
CAND_FILE=TBL/'02_2023断面ADT_only_物理QC逐日记录.csv'
if CAND_FILE.exists() and not FORCE_QC_REPROCESS:
 prev=pd.read_csv(CAND_FILE)
else:prev=pd.DataFrame()
known=set(prev.date.astype(str)) if len(prev) else set()
allrows=[]
for j,(day,g) in enumerate(part_sec.groupby('date',sort=True),1):
 day=str(day)
 if day in known:continue
 meta=part_status.loc[part_status.date==day].iloc[0]
 axfile=PART/(day.replace('-','')+'.npz')
 try:
  a=load_axis(axfile)
  if len(a)<20:raise ValueError('部分轴不足20点')
  lo,la,adt=read_adt_field(Path(meta.source_nc),day);F=fields(lo,la,adt)
  for r in g.itertuples(index=False):
   row={'date':day,'section':r.section,'r72_candidate_safe':bool(r.safe),'n_safe_intersections':int(r.n_safe_intersections),
        'distance_km':float(r.distance_km) if pd.notna(r.distance_km) else np.nan,
        'lon':float(r.selected_lon) if pd.notna(r.selected_lon) else np.nan,
        'lat':float(r.selected_lat) if pd.notna(r.selected_lat) else np.nan,
        'source_nc':str(meta.source_nc)}
   try:
    reasons=[]
    if not r.safe:reasons.append('no_safe_geometric_intersection')
    if REQUIRE_UNIQUE_INTERSECTION and int(r.n_safe_intersections)!=1:reasons.append('ambiguous_intersection')
    if len(reasons)==0:
     physical=section_qc_physics(F,a,r.selected_lon,r.selected_lat)
     row.update(physical)
     for metric in ['fine_ridge_ratio','coarse_ridge_ratio','geostrophic_dir_cos']:
      if not np.isfinite(physical[metric]) or physical[metric]<THRESHOLDS[r.section][metric]:reasons.append('low_'+metric)
    row['pass_adt_physical_qc']=len(reasons)==0
    row['reject_reason']=';'.join(reasons)
   except Exception as e:
    row['pass_adt_physical_qc']=False;row['reject_reason']='physics_exception_'+repr(e)
   allrows.append(row)
 except Exception as e:
  for r in g.itertuples(index=False):
   allrows.append({'date':day,'section':r.section,'r72_candidate_safe':bool(r.safe),'pass_adt_physical_qc':False,'reject_reason':'nc_or_axis_error_'+repr(e)})
 if len(allrows)>=90 or j==part_sec.date.nunique():
  prev=pd.concat([prev,pd.DataFrame(allrows)],ignore_index=True)
  prev=prev.drop_duplicates(['date','section'],keep='last').sort_values(['date','section'])
  prev.to_csv(CAND_FILE,index=False,encoding='utf-8-sig');allrows.clear()
 if j%30==0:print('ADT physics QC',j,'/274',flush=True)
qc2023=pd.read_csv(CAND_FILE)
assert len(qc2023)==1644 and qc2023.duplicated(['date','section']).sum()==0, 'QC输出日期/断面不完整'
qc2023['pass_adt_physical_qc']=qc2023.pass_adt_physical_qc.astype(bool)
qc_summary=qc2023.groupby('section',sort=False).agg(geometric_candidates=('r72_candidate_safe','sum'),adt_physical_qc_pass=('pass_adt_physical_qc','sum')).reset_index()
qc_summary.to_csv(TBL/'03_各断面物理QC通过统计.csv',index=False,encoding='utf-8-sig')
display(qc_summary)


In [ ]:
# Cell 5 — 冻结QC后再评估 GLORYS（仅敏感性分析；不能把筛选结果重新反馈到QC）
# 计算完整ADT参考的历史配对，和额外2023年ADT-only QC后的配对。

def cross2(a,b):return float(a[0]*b[1]-a[1]*b[0])
def all_intersections(axis,ends):
 p1,p2=np.array(ends[0],float),np.array(ends[1],float)
 # 投影到该断面纬度中心附近的局地公里坐标
 phi=(p1[1]+p2[1])/2
 q=at_km(np.array([p1,p2]),phi); R=q[1]-q[0]
 aa=at_km(axis,phi);hits=[]
 for i in range(len(axis)-1):
  p=aa[i];v=aa[i+1]-p;den=cross2(v,R)
  if abs(den)<1e-10:continue
  t=cross2(q[0]-p,R)/den;u=cross2(q[0]-p,v)/den
  if -1e-8<=t<=1+1e-8 and -1e-8<=u<=1+1e-8:
   d=np.linalg.norm(v)
   hit=axis[i]+float(np.clip(t,0,1))*(axis[i+1]-axis[i])
   dkm=float(np.linalg.norm(at_km(hit,phi)-q[0]))
   if not any(abs(dkm-h['distance_km'])<.05 for h in hits):hits.append({'distance_km':dkm,'point':hit})
 return hits

selected=qc2023.loc[qc2023.pass_adt_physical_qc].copy()
rows=[]
if USE_GLORYS_AFTER_FROZEN_QC:
 for i,(day,g) in enumerate(selected.groupby('date'),1):
  p=GLORYS/(day.replace('-','')+'.npz')
  if not p.exists():
   for r in g.itertuples(index=False):rows.append({'date':day,'section':r.section,'error':'missing_glorys_npz'})
   continue
  a=load_axis(p)
  for r in g.itertuples(index=False):
   hits=all_intersections(a,SECTIONS[r.section])
   if len(hits)==0:
    rows.append({'date':day,'section':r.section,'error':'no_glorys_intersection'});continue
   # 与论文原监测断面定义保持一致：沿轨迹首次断面交点
   gl=hits[0]['distance_km']
   rows.append({'date':day,'section':r.section,'source':'ADT_partial_QC_pass_sensitivity',
      'adt_km':float(r.distance_km),'glorys_km':gl,'difference_km':gl-float(r.distance_km),
      'glorys_intersections':len(hits),'error':''})
  if i%30==0:print('Post-QC GLORYS diagnostic:',i,'/',selected.date.nunique())
comp=pd.DataFrame(rows)
if not len(comp):comp=pd.DataFrame(columns=['date','section','adt_km','glorys_km','difference_km','error'])
comp.to_csv(TBL/'04_仅物理QC通过后的ADT_vs_GLORYS断面敏感性.csv',index=False,encoding='utf-8-sig')
# 不完整流轴主结果仍为R7.0全部10,279天；这里只讨论2023年断面补充样本
cover=[]
for section in SECTIONS:
 n_full=int(len(full_sections[(full_sections.section==section)&(full_sections.date.astype(str).str.startswith('2023'))&full_sections.distance_km.notna()]))
 n_pass=int(selected[selected.section==section].date.nunique())
 n_pairs=int(len(comp[(comp.section==section)&comp.error.fillna('').eq('')]))
 cover.append({'section':section,'full_2023_ADT_days':n_full,'partial_2023_QC_pass_days':n_pass,
               'candidate_combined_days':n_full+n_pass,'qc_pass_glorys_pairs':n_pairs,
               'main_full_axis_days_unchanged':91,'publishable_as_independent_full_axis':False})
cover=pd.DataFrame(cover)
cover.to_csv(TBL/'05_2023断面可用性_物理QC敏感性.csv',index=False,encoding='utf-8-sig')
display(cover)


In [ ]:
# Cell 6 — 2023月份覆盖诊断图＋全年独立性和科学边界报告
plt.rcParams.update({'font.size':10,'axes.grid':True,'grid.alpha':0.22})
fig,ax=plt.subplots(figsize=(9.8,4.5),dpi=160)
x=np.arange(len(cover));w=.27
ax.bar(x-w,cover.full_2023_ADT_days,width=w,label='Original complete ADT tracks')
ax.bar(x,cover.partial_2023_QC_pass_days,width=w,label='ADT-only QC-approved partial sections')
ax.bar(x+w,cover.candidate_combined_days,width=w,label='Combined sensitivity potential')
ax.axhline(365,color='gray',lw=.7,ls=':')
ax.set_xticks(x,cover.section);ax.set(xlabel='Section',ylabel='Days (2023)',title='ADT reference coverage — incomplete trajectories remain sensitivity-only')
ax.legend(fontsize=8);fig.tight_layout();fig.savefig(FIG/'Fig01_2023_SectionPhysicalQC_Coverage.png',dpi=240);fig.savefig(FIG/'Fig01_2023_SectionPhysicalQC_Coverage.pdf');plt.close(fig)

q=qc2023.copy();q['month']=pd.to_datetime(q.date).dt.month
pivot=q.pivot_table(index='month',columns='section',values='pass_adt_physical_qc',aggfunc='sum').reindex(range(1,13)).fillna(0)
fig,ax=plt.subplots(figsize=(10,4),dpi=160)
for sec in SECTIONS:ax.plot(pivot.index,pivot.get(sec,pd.Series(0,index=pivot.index)),marker='.',label=sec)
ax.set(xlabel='Month (2023)',ylabel='ADT-only QC-passed partial sections (days)',title='Monthly availability of 2023 incomplete ADT trajectories')
ax.set_xticks(range(1,13));ax.legend(ncol=6);fig.tight_layout();fig.savefig(FIG/'Fig02_2023_QC_Passed_ByMonth.png',dpi=240);fig.savefig(FIG/'Fig02_2023_QC_Passed_ByMonth.pdf');plt.close(fig)

status={'run_date_utc':pd.Timestamp.now(tz='UTC').isoformat(),'source_r72_zip_sha256':hashlib.sha256(R72_ZIP.read_bytes()).hexdigest(),
 'n_original_r72_failed_tracks':274,'r72_partial_section_candidates':int(part_sec.safe.sum()),
 'n_qc_pass_intersections':int(selected.shape[0]),'qc_uses_glorys':False,'glorys_analyzed_only_after_qc':USE_GLORYS_AFTER_FROZEN_QC,
 'calibration_years':list(CALIBRATION_YEARS),'calibration_quantile':THRESHOLD_QUANTILE,
 'quality_gate_warning':'No physically independent vessel/ADCP validation; ADT front is not 50m velocity core',
 'no_change_to_r70_r72_main_10279_full_axis_result':True,
 'forecast_evaluation_completed':False,
 'limits':['Full 2023 complete-axis count remains 91','Partial sections are sensitivity-only and can contain reference extraction errors',
           'Thresholds calibrated on ADT full tracks, not an external perfect truth','The same satellite ADT system produced both candidate and QC fields',
           '2018-2022 reference sampling and residual correlations cannot establish observation independence',
           'GLORYS reanalysis may assimilate altimetry','Checkpoint-based 30/60/90d forecast paths still missing']}
(REP/'R73_科学边界与运行状态.json').write_text(json.dumps(status,ensure_ascii=False,indent=2),encoding='utf-8')
(REP/'R73_中文审稿说明.txt').write_text(
 'R7.3只将ADT-only QC通过的2023不完整流轴断面交点作为附加敏感性证据；不能视为完整卫星参考流轴，也不增加R7.0的10,279条完整流轴。'
 '筛选在对比GLORYS之前完成，依据2018–2022完整ADT流轴的物理梯度脊线和方向统计阈值。'
 '没有导入真实模型30/60/90天预测轨迹，因此预测vsADT尚未完成。',encoding='utf-8')
archive=ROOT/'R73_上传给ChatGPT_2023ADT断面独立物理QC.zip'
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED,compresslevel=6) as z:
 for folder in (TBL,REP,FIG):
  for p in sorted(folder.rglob('*')):
   if p.is_file():z.write(p,p.relative_to(OUT))
with zipfile.ZipFile(archive) as z:assert z.testzip() is None
hashval=hashlib.sha256(archive.read_bytes()).hexdigest()
Path(str(archive)+'.sha256').write_text(hashval+'  '+archive.name+'\n',encoding='utf-8')
print('R7.3已打包：',archive, '\nSHA256:',hashval)
display(FileLink(str(archive)))


### 解释和后续优先事项

R7.3 只有局地卫星物理一致性 QC，没有将失败全轨迹转换为成功。R7.0 的完整参考轴监测验证结果仍为 10,279 天；R7.2 共同支持区间 RMSE 42.04 km 不因这次部分轴分析而改变。

必须尽快完成 R6.5 已冻结权重的真实预测轨迹导出，并按 `issue_date + lead_days = target_date` 与独立 ADT 参考配对。正式论文的 30/60/90 天预测结果不能由训练误差或监测轨迹反算。